In [ ]:
!pip install openai python-dotenv streamlit
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
up to date, audited 23 packages in 2s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠸

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI


# Load environment variables from .env file so no one can see your API key
load_dotenv()
# Initialize OpenAI API client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def generate_response(messages):
    """Simple function to interact with OpenAI's API."""
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
messages = [
    {"role": "system", "content": "Talk like a pirate."},
    {
        "role": "user",
        "content": "How do I check if a Python object is an instance of a class?",
    },
]
response = generate_response(messages)
print(response)

Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


### 2. Creating the Streamlit Interface

Now that we know how to generate response using OpenAI APIs, lets setup a web interface using [Streamlit](https://docs.streamlit.io) similar to ChatGPT's interface.

We will use `localtunnel` for tunnelling so the web server running in Google Colab can be accessed from our browser as well.

Let's first write the following contents to `app_openai.py` for streamlit to run.

In [ ]:
%%writefile app_openai.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI


# Load environment variables from .env file so no one can see your API key
load_dotenv()

# Initialize OpenAI API client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def generate_response(messages):
    """Simple function to interact with OpenAI's API."""
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Title
st.title("🤖 My own ChatGPT!")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "system",
            "content": "Talk like you are explaining to a 5 year old.",
        },
    ]

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat input
if prompt := st.chat_input("What would you like to know?"):
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Display user message
    with st.chat_message("user"):
        st.markdown(prompt)

    # Generate and display AI response
    with st.chat_message("assistant"):
        response = generate_response(st.session_state.messages)
        st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})

Overwriting app_openai.py


Finally, lets run the streamlit app! You will get a link to open the app where you can enter the IP address to access it and start chatting!

> **NOTE**: Stop the below cell once you are done using your app.

In [ ]:
!streamlit run app_openai.py &>logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.124.250.55
⠙your url is: https://honest-streets-count.loca.lt


In [ ]:
!pip install transformers torch streamlit
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
up to date, audited 23 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠹

In [ ]:
import torch
import transformers

pipeline = transformers.pipeline("text-generation", model="microsoft/Phi-4-mini-instruct", torch_dtype=torch.bfloat16, device_map="auto")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:
def generate_response(messages: list[dict]):
    return pipeline(messages, max_new_tokens=500, return_full_text=False)[0]["generated_text"]


messages = [
    {"role": "system", "content": "You are an experienced Content Creator with more than 1 Million followers on Instagram. You always respond in bullet points."},
    {"role": "user", "content": "What are some trending content ideas for travel-related reels?"},
    {
        "role": "assistant",
        "content": "Here are some viral travel content ideas for reels: 1. Hidden gem discoveries with dramatic reveals. 2. Travel hacks for packing and planning. 3. Before/after transformations of travel spots (sunrise vs sunset). 4. Local food and culture experiences with trending music.",
    },
    {"role": "user", "content": "How can I improve my travel content engagement rate?"},
]

response = generate_response(messages)
print(response)

- Use high-quality, visually appealing images and videos.
- Incorporate trending music and hashtags.
- Engage with your audience by asking questions and encouraging comments.
- Collaborate with other travel influencers.
- Share personal stories and experiences to create a connection with your audience.
- Post consistently and at optimal times for your audience.
- Use Instagram Stories and Reels to keep your audience engaged.
- Run contests or giveaways to encourage interaction.
- Use Instagram's analytics to understand your audience and tailor your content accordingly.


In [ ]:
%%writefile app_huggingface.py
import streamlit as st
import torch
import transformers


@st.cache_resource
def load_model():
    """Load the model and tokenizer"""
    return transformers.pipeline("text-generation", model="microsoft/Phi-4-mini-instruct", torch_dtype=torch.bfloat16, device_map="auto")


# Load the model
pipeline = load_model()


def generate_response(messages):
    """Generate a response using the model"""
    try:
        response_text = pipeline(messages, max_new_tokens=500, return_full_text=False)[0]["generated_text"]
        return response_text if response_text else "I'm not sure how to respond to that."
    except Exception as e:
        return f"Error: {str(e)}"


# Title
st.title("🤖 My Coding Assignment Helper!")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "system",
            "content": "You are an expert coder with more than 10 years of experience in Python.",
        },
    ]

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat input
if prompt := st.chat_input("What would you like to know?"):
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Display user message
    with st.chat_message("user"):
        st.markdown(prompt)

    # Generate and display AI response
    with st.chat_message("assistant"):
        response = generate_response(st.session_state.messages)
        st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})

Overwriting app_huggingface.py


In [ ]:
!streamlit run app_huggingface.py &>logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.124.250.55
⠙⠹your url is: https://funny-beds-end.loca.lt
